# 17.15 品質與行為保留了多少？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**平均logit只差0.01，就保證greedy序列一樣嗎？

壓縮後重跑同一批內容與行爲檢查。小logits誤差也可能跨過argmax邊界，改變格式、EOS或事實答案；不能只報weight MAE。

**把直覺寫成數學：**比較原版/量化版的BPB、任務正確率、格式、誠實、安全、成本。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
a = torch.tensor([1.0, 1.005])
b = torch.tensor([1.01, 1.005])
print("平均誤差", (a - b).abs().mean().item(), "argmax", a.argmax().item(), b.argmax().item())
assert a.argmax() != b.argmax()

**如何讀結果：**這個邊界反例說明必須實際生成與評價；安全檢查仍只支持測到的情境。

**只改一件事：**只把原本分數差擴大到1，觀察小誤差較不易翻轉。
